In [1]:
import numpy as np
import torch
import tqdm

from pathlib import Path

from controller.optim_cntrlr import ManualManifoldPlantDynamics
from episodes.traj import Trajectory
from src.episodes.episode import Episode

from src.manifolds.sn_mfld import HypersphereManifold
from src.episodes.episode import Episode
from src.controller.optim_cntrlr import OptimizationController

r = np.random.default_rng(42)  # for reproducibility

In [2]:
EPISODE_DATA_DIR = Path("../data/episodes/hypersphere")

HORIZON_STEPS = 3
HORIZON_STEP_DT = 0.2

STATE_COST_FACTOR = 10.0
INPUT_COST_FACTOR = 0.25

In [3]:
episodes = []
for child in EPISODE_DATA_DIR.iterdir():
    if not child.is_file() or child.name == ".gitkeep":
        continue
    episodes.append(Episode.load(child))

In [4]:
def input_cost(time: float, inputs: np.ndarray) -> float:
    return INPUT_COST_FACTOR * (inputs.transpose() @ inputs).item()

def state_cost(time: float, pos: np.ndarray, vel: np.ndarray, traj: Trajectory, manifold: HypersphereManifold) -> float:
    current_traj_extrinsic = traj.extrinsic_at_t(time)
    current_pos_extrinsic = manifold.to_extrinsic(manifold.default_chart, torch.tensor(pos)).numpy()

    err = current_traj_extrinsic - current_pos_extrinsic
    state_err = STATE_COST_FACTOR * (err.transpose() @ err).item()

    return state_err

In [5]:
for i, episode in tqdm.tqdm(enumerate(episodes), desc="Episodes", total=len(episodes)):
    hs_radius = episode.params["hs_radius"].item()  # needs to be a float
    hs_dim = episode.params["hs_dim"]
    sample_time = episode.params["sample_time"]
    traj_idx = episode.params["traj_idx"]
    episode_idx = episode.params["episode_idx"]

    hypersphere_manifold = HypersphereManifold(hs_dim, hs_radius)
    dynamics = ManualManifoldPlantDynamics(
        hypersphere_manifold,
        (episode.initial_pos, episode.initial_vel))

    num_timesteps = episode.target_traj.time.shape[0]

    optim_controller = OptimizationController(dynamics,
                                              lambda t, pos, vel: state_cost(t, pos, vel, episode.target_traj,
                                                                             hypersphere_manifold),
                                              input_cost, HORIZON_STEPS, HORIZON_STEP_DT)

    for j in tqdm.tqdm(range(num_timesteps), desc="Timesteps", total=num_timesteps, leave=True):
        optimal_u = optim_controller.generate_optimal_controls()

        print(optimal_u)
        print(optimal_u.shape)

        result = dynamics.step(sample_time, optimal_u)

        current_traj_extrin = episode.target_traj.extrinsic_at_t(result.time)
        current_pos_extrin = hypersphere_manifold.to_extrinsic(hypersphere_manifold.default_chart, torch.tensor(result.pos)).numpy()

        print(f"current_traj_extrin: {current_traj_extrin}")
        print(f"current_pos_extrin: {current_pos_extrin}")
        print(f"err: {current_traj_extrin - current_pos_extrin}")

        if j == 50:
            break
    break

    pass

Timesteps:   0%|          | 1/242 [00:06<24:38,  6.13s/it]

result.x: [-3.26550521e-07 -1.96916296e-07 -1.24802921e-08 -2.46339176e-07
 -2.39434641e-07  5.69779064e-08 -6.44688412e-08  1.20894057e-07
  1.37171373e-07]
optimal horizon_inputs: [[-3.26550521e-07 -1.96916296e-07 -1.24802921e-08]
 [-2.46339176e-07 -2.39434641e-07  5.69779064e-08]
 [-6.44688412e-08  1.20894057e-07  1.37171373e-07]]
[-3.26550521e-07 -1.96916296e-07 -1.24802921e-08]
(3,)
current_traj_extrin: [ 0.17816921 -0.42624322 -0.78759933  0.40774956]
current_pos_extrin: [ 0.5906235  -0.44813606 -0.43203506  0.5135014 ]
err: [-0.41245429  0.02189285 -0.35556427 -0.10575184]



Timesteps:   1%|          | 2/242 [00:06<11:14,  2.81s/it]

result.x: [-3.26550521e-07 -1.96916296e-07 -1.24802921e-08 -2.46339176e-07
 -2.39434641e-07  5.69779064e-08 -6.44688412e-08  1.20894057e-07
  1.37171373e-07]
optimal horizon_inputs: [[-3.26550521e-07 -1.96916296e-07 -1.24802921e-08]
 [-2.46339176e-07 -2.39434641e-07  5.69779064e-08]
 [-6.44688412e-08  1.20894057e-07  1.37171373e-07]]
[-3.26550521e-07 -1.96916296e-07 -1.24802921e-08]
(3,)
current_traj_extrin: [ 0.11524555 -0.44230723 -0.81003422  0.36732465]
current_pos_extrin: [ 0.5743942  -0.47423577 -0.4274303   0.51232314]
err: [-0.45914868  0.03192854 -0.38260391 -0.14499849]



Timesteps:   1%|          | 3/242 [00:07<06:58,  1.75s/it]

...


Timesteps:   5%|▍         | 12/242 [04:08<1:57:37, 30.69s/it]

result.x: [ 2.60475212e+00 -4.86109162e-01 -4.86108985e-01  1.74927470e+00
 -1.51150245e-07 -4.86108988e-01 -6.44688412e-08  1.20894003e-07
  1.37171319e-07]
optimal horizon_inputs: [[ 2.60475212e+00 -4.86109162e-01 -4.86108985e-01]
 [ 1.74927470e+00 -1.51150245e-07 -4.86108988e-01]
 [-6.44688412e-08  1.20894003e-07  1.37171319e-07]]
[ 2.60475212 -0.48610916 -0.48610898]
(3,)
current_traj_extrin: [-0.36639062 -0.50937837 -0.77846247  0.01696491]
current_pos_extrin: [ 0.29504606 -0.6337598  -0.53148645  0.4783497 ]
err: [-0.66143668  0.12438142 -0.24697602 -0.46138477]


Episodes:   0%|          | 0/33 [04:42<?, ?it/s]

Unexpected exception formatting exception. Falling back to standard exception



Traceback (most recent call last):
  File "/home/samuel/Documents/School/UWaterloo_MASc/Courses/ECE_657D/project/ece657d-geo-dyn-learning-tradeoffs/.venv/lib/python3.14/site-packages/IPython/core/interactiveshell.py", line 3747, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_247558/1618596641.py", line 21, in <module>
    optimal_u = optim_controller.generate_optimal_controls()
  File "/home/samuel/Documents/School/UWaterloo_MASc/Courses/ECE_657D/project/ece657d-geo-dyn-learning-tradeoffs/src/controller/optim_cntrlr.py", line 195, in generate_optimal_controls
    result = minimize(step_cost, guess_horizon_inputs.flatten(), method="bfgs")
  File "/home/samuel/Documents/School/UWaterloo_MASc/Courses/ECE_657D/project/ece657d-geo-dyn-learning-tradeoffs/.venv/lib/python3.14/site-packages/scipy/optimize/_minimize.py", line 779, in minimize
    res = _minimize_bfgs(fun, x0, args, jac, callback, **